# Table 1. Agreement of the Python port with the released IndPenSim dataset

Reference (MATLAB): the released 100-batch dataset of Goldrick et al. (2019); batches 1-30 run on the recipe with
Raman-based control off, batches 61-90 under Raman-based PAA control, batches 91-100 contain faults.
pyIndPenSim: 100 batches (seeds 2000-2099) with Raman-based control off and on.

Inputs (data/): table1_reference.npz   yields, 12 h off-line PAA and trajectories of the released batches (tools/extract_table1_reference.py) raman_onoff_paa.npz    yields and PAA on the 12 h grid of the 100 pyIndPenSim batches, control off and on table1_pool300.npz     trajectories of a pool of 300 pyIndPenSim batches, from which 90 are matched to the 90 fault-free released batches on the penicillin trajectory

In [1]:
import sys; sys.path.insert(0, '..')          # repository root, for paths.py
from paths import DATA, OUTD                   # data/ and output/

In [2]:
import numpy as np
from scipy import stats
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

ref = np.load(DATA + 'table1_reference.npz', allow_pickle=True)
sim = np.load(DATA + 'raman_onoff_paa.npz')
pool = np.load(DATA + 'table1_pool300.npz')
GS = sim['grid']                                    # 12 h grid, 12-228 h, as in the released off-line record


def cohen_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (b.mean() - a.mean()) / sp


L = ['Table 1. Agreement of the Python port with the released IndPenSim dataset', '',
     'Batch yield, mean +- SD (kg)']
for name, sl, key in (('Raman off, recipe-driven', slice(0, 30), 'off'), ('Raman on, PAT control', slice(60, 90), 'on')):
    r = ref['yield_kg'][sl]; s = sim[key][np.isfinite(sim[key])]; d = s.mean() - r.mean()
    L.append(f'  {name:<26s} reference {r.mean():,.0f} +- {r.std(ddof=1):,.0f} ({len(r)})   '
             f'pyIndPenSim {s.mean():,.0f} +- {s.std(ddof=1):,.0f} ({len(s)})   {d:+,.0f} ({100 * d / r.mean():+.2f} %)   '
             f"Cohen's d {cohen_d(r, s):+.3f} . KS p = {stats.ks_2samp(r, s).pvalue:.3f} . "
             f'MWU p = {stats.mannwhitneyu(r, s).pvalue:.3f}')

In [3]:
# PAA: both sides on the same 12 h grid; the released off-line samples are interpolated onto it
L += ['', 'PAA, mean over the batch (mg/L), 12 h sampling']
for name, b0, key in (('Raman off, recipe-driven', 0, 'off'), ('Raman on, PAT control', 60, 'on')):
    rv = [np.interp(GS, ref['paa_times'][b], ref['paa_off_mgL'][b]) for b in range(b0, b0 + 30)]
    r = np.mean(rv); s = np.nanmean(sim[f'{key}_pgrid']); d = s - r
    L.append(f'  {name:<26s} reference {r:,.0f} (30)   pyIndPenSim {s:,.0f} (100)   {d:+,.0f} ({100 * d / r:+.2f} %)')

In [4]:
# Process variables: 90 pool batches matched to the 90 fault-free released batches on P(t) (Hungarian assignment)
cost = cdist(ref['traj_P'], pool['traj_P'], 'euclidean'); row, col = linear_sum_assignment(cost)
matched = col[np.argsort(row)]
L += ['', 'Process variables, mean over the batch (n = 90 vs 90, matched on the penicillin trajectory)',
      f'  {"":28s}{"reference":>12s}{"pyIndPenSim":>13s}{"mean abs. diff.":>17s}{"% of range":>12s}']
for k, name in (('S', 'Substrate, S (g/L)'), ('DO2', 'Dissolved O2 (mg/L)'), ('pH', 'pH'),
                ('T', 'Temperature, T (K)'), ('V', 'Volume, V (L)')):
    rm = np.nanmean(ref[f'traj_{k}'], 0); pm = np.nanmean(pool[f'traj_{k}'][matched], 0)
    mae = np.nanmean(np.abs(rm - pm)); rng = np.nanmax(rm) - np.nanmin(rm)
    L.append(f'  {name:28s}{np.nanmean(rm):12,.3f}{np.nanmean(pm):13,.3f}{mae:17,.3f}{100 * mae / rng:11.1f} %')

txt = '\n'.join(L); print(txt); open(OUTD + 'table1.txt', 'w').write(txt + '\n')

Table 1. Agreement of the Python port with the released IndPenSim dataset

Batch yield, mean +- SD (kg)
  Raman off, recipe-driven   reference 2,913 +- 768 (30)   pyIndPenSim 2,894 +- 802 (100)   -18 (-0.63 %)   Cohen's d -0.023 . KS p = 0.856 . MWU p = 0.836
  Raman on, PAT control      reference 3,484 +- 308 (30)   pyIndPenSim 3,481 +- 487 (100)   -3 (-0.08 %)   Cohen's d -0.006 . KS p = 0.278 . MWU p = 0.709

PAA, mean over the batch (mg/L), 12 h sampling
  Raman off, recipe-driven   reference 1,786 (30)   pyIndPenSim 1,788 (100)   +1 (+0.06 %)
  Raman on, PAT control      reference 1,218 (30)   pyIndPenSim 1,216 (100)   -1 (-0.11 %)

Process variables, mean over the batch (n = 90 vs 90, matched on the penicillin trajectory)
                                 reference  pyIndPenSim  mean abs. diff.  % of range
  Substrate, S (g/L)                 2.347        2.364            0.138        1.0 %
  Dissolved O2 (mg/L)               12.468       12.713            0.245        3.8 %
  pH 

1253